# 🧠 LLM Non-Instruction Fine-Tuning: A Learning Journey

> **Philosophy of this notebook:**  
> We intentionally show the *wrong* approach first, hit the real error, understand *why* it fails,  
> then evolve the code to the correct solution. Each section builds on the previous.

---

## What is Non-Instruction Fine-Tuning?

| Type | Dataset Format | Goal |
|------|---------------|------|
| **Non-Instruction (this notebook)** | Raw text (books, papers, articles) | Teach model domain knowledge |
| Instruction Fine-Tuning | `{instruction, response}` pairs | Teach model to follow commands |
| Preference Fine-Tuning | `{prompt, chosen, rejected}` | Align model behavior via RLHF/DPO |

**Our goal:** Take a pre-trained LLM and make it fluent in a specific domain (e.g., medical/pharmaceutical text from PDFs).

---

## Frameworks Available for Fine-Tuning

| Framework | Best For |
|-----------|----------|
| `transformers` + `trl` | Learning, custom control |
| `Unsloth` | Fastest QLoRA, great for Colab |
| `Axolotl` | Production, config-file driven |
| `LlamaFactory` | GUI + CLI, beginner friendly |
| `DeepSpeed` | Multi-GPU, large models |

> 💡 We use `transformers` + `peft` + `trl` here — most transparent for learning.

---
# PART 1 — Install & Imports

In [ ]:
# Install all required libraries
# bitsandbytes  → 4-bit/8-bit quantization (needed for QLoRA)
# peft          → LoRA and other parameter-efficient methods
# accelerate    → multi-device training abstraction
# trl           → Trainer wrappers built on top of transformers
# PyMuPDF       → PDF text extraction (fitz)

!pip install -q -U transformers peft bitsandbytes accelerate trl PyMuPDF datasets

---
# PART 2 — Prepare the Dataset

For non-instruction fine-tuning, your data is just **raw text** — no special prompt/response format needed.  
The model learns by predicting the next token on your domain text.

### Option A: Use a HuggingFace Dataset (Public)
Great for experimentation before using your own data.

### Option B: Extract from Your Own PDF (Domain-Specific)
This is what makes your fine-tuned model actually useful.

In [ ]:
# ── Option A: Public HuggingFace Dataset ──────────────────────────────────────
# Uncomment whichever dataset suits your domain:

# General stories (tiny, great for testing on free Colab)
# from datasets import load_dataset
# dataset = load_dataset("roneneldan/TinyStories", split="train")

# Medical/Biomedical
# dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer", split="train")

# Scientific papers
# dataset = load_dataset("armanc/scientific_papers", "pubmed", split="train")

# General web text
# dataset = load_dataset("Skylion007/openwebtext", split="train")

print("Uncomment the dataset you want to use above.")

In [ ]:
# ── Option B: Extract Text from Your Own PDF ──────────────────────────────────
# Upload your PDF to Colab first: Files panel → Upload

import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path: str) -> list[str]:
    """
    Extract text page-by-page from a PDF.
    Returns a list where each element is the text of one page.
    """
    pages = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:  # skip blank pages
                pages.append(text)
    return pages


# ── CHANGE THIS PATH to your uploaded PDF ──
pdf_path = "/content/Metformin.pdf"
pages = extract_text_from_pdf(pdf_path)

print(f"Extracted {len(pages)} pages")
print("\nSample (first 300 chars of page 1):")
print(pages[0][:300])

In [ ]:
# ── Attempt 1: Naive chunking — split on double newlines ──────────────────────
#
# ❌ PROBLEM with this approach:
#   - PDFs often don't have clean double newlines between paragraphs
#   - Splits mid-sentence at arbitrary points
#   - Creates chunks of wildly unequal length (some 5 words, some 5000 words)
#   - The tokenizer will truncate large chunks, losing information
#   - Short chunks carry very little signal for the model to learn from
#
# We run this to SEE the problem, not because it's correct.

import re

def chunk_naive(pages: list[str]) -> list[str]:
    chunks = []
    for page_text in pages:
        for chunk in re.split(r'\n\s*\n', page_text):
            clean = chunk.strip()
            if len(clean) > 30:  # skip tiny fragments
                chunks.append(clean)
    return chunks

naive_chunks = chunk_naive(pages)

# Inspect the problem
lengths = [len(c) for c in naive_chunks]
print(f"Total chunks: {len(naive_chunks)}")
print(f"Min length : {min(lengths)} chars")
print(f"Max length : {max(lengths)} chars")
print(f"Mean length: {sum(lengths)//len(lengths)} chars")
print("\n⚠️  Notice the huge variance — this is the problem!")

In [ ]:
# ── Attempt 2: Better chunking — fixed size with overlap ──────────────────────
#
# ✅ WHY THIS IS BETTER:
#   - Consistent chunk sizes → predictable tokenization
#   - Overlap (50 chars) → context is NOT cut abruptly at boundaries
#   - No external dependency (no LangChain needed for this simple case)
#
# chunk_size = 400 chars ≈ ~100 tokens (rule of thumb: 1 token ≈ 4 chars)
# This fits comfortably inside our max_length=512 tokenizer setting.

def chunk_with_overlap(pages: list[str], chunk_size: int = 400, overlap: int = 50) -> list[str]:
    """
    Merge all pages into one text, then slide a window of `chunk_size` chars
    across it, stepping by (chunk_size - overlap) each time.
    """
    full_text = "\n".join(pages)  # join all pages
    full_text = re.sub(r'\s+', ' ', full_text).strip()  # normalize whitespace

    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(full_text), step):
        chunk = full_text[start : start + chunk_size].strip()
        if len(chunk) > 30:
            chunks.append(chunk)
    return chunks


paragraphs = chunk_with_overlap(pages, chunk_size=400, overlap=50)

lengths = [len(c) for c in paragraphs]
print(f"Total chunks: {len(paragraphs)}")
print(f"Min length : {min(lengths)} chars")
print(f"Max length : {max(lengths)} chars")
print(f"Mean length: {sum(lengths)//len(lengths)} chars")
print("\n✅ Much more consistent — great for training!")
print("\nSample chunk:")
print(paragraphs[2])

In [ ]:
# ── Build HuggingFace Dataset from our chunks ─────────────────────────────────

from datasets import Dataset

data = [{"text": chunk} for chunk in paragraphs]
dataset = Dataset.from_list(data)

# Always split into train/validation
# Without validation, you have NO way to detect overfitting!
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train samples : {len(train_dataset)}")
print(f"Eval  samples : {len(eval_dataset)}")
print(f"\nFirst sample  : {train_dataset[0]['text'][:200]}...")

---
# PART 3 — Tokenization

The tokenizer converts raw text into integer token IDs that the model understands.  
For **causal language modeling (CLM)**, the `labels` are just the `input_ids` shifted by one —  
the model tries to predict token N+1 given tokens 0..N.

In [ ]:
# ── Choose Your Model ─────────────────────────────────────────────────────────
#
# Free Colab (T4, ~15GB VRAM):
#   TinyLlama 1.1B  ← we use this
#   Phi-2 2.7B
#
# Colab Pro / A100 (40GB VRAM):
#   meta-llama/Llama-3.1-8B
#   meta-llama/Llama-2-7b-hf
#   mistralai/Mistral-7B-v0.3
#
# Rule of thumb: 1B params ≈ 2GB in fp16, ≈ 0.5GB in 4-bit

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
print(f"Using model: {model_name}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Most causal models don't have a pad token by default
# (they were trained to generate, not to pad).
# We set it to eos_token so the tokenizer doesn't crash.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size     : {tokenizer.vocab_size}")
print(f"Pad token      : {tokenizer.pad_token!r}")
print(f"EOS token      : {tokenizer.eos_token!r}")

In [ ]:
# ── Tokenization Attempt 1: Naive labels — WRONG ──────────────────────────────
#
# ❌ PROBLEM: padding tokens also get trained on!
#
# When we pad shorter sequences to max_length=512,
# those padding positions have token_id = pad_token_id.
# If labels = input_ids (a direct copy), the model is penalized
# for NOT predicting padding tokens — which is meaningless noise.
#
# This INFLATES the loss with garbage signal and slows/corrupts learning.

def tokenize_naive(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()  # ❌ includes padding in loss
    return tokens

print("Naive tokenization defined — but we will NOT use it.")
print("See the corrected version below.")

In [ ]:
# ── Tokenization Attempt 2: Mask padding in labels — CORRECT ─────────────────
#
# ✅ FIX: Replace pad token positions in `labels` with -100
#
# PyTorch's CrossEntropyLoss ignores positions where label == -100.
# So the model only learns from real text tokens, not from padding.
#
# This is the standard approach in ALL serious fine-tuning code.

def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    # Mask padding tokens in labels so they don't contribute to loss
    labels = [
        [
            token_id if token_id != tokenizer.pad_token_id else -100
            for token_id in ids
        ]
        for ids in tokens["input_ids"]
    ]
    tokens["labels"] = labels
    return tokens


tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval  = eval_dataset.map(tokenize_fn,  batched=True, remove_columns=["text"])

print(tokenized_train)
print("\nColumns:", tokenized_train.column_names)
print("\nSanity check — unique label values (should include -100):")
import collections
sample_labels = tokenized_train[0]["labels"]
print(f"  -100 count: {sample_labels.count(-100)}  (padding positions masked)")
print(f"  Real token count: {sum(1 for l in sample_labels if l != -100)}")

---
# PART 4 — Attempt 1: Full Fine-Tuning (Will Crash)

Before using any tricks, let's try the simplest approach:  
**load the model and train ALL its parameters.**

This is called **Full Fine-Tuning** — every weight in the model gets updated.

Spoiler: on a free Colab T4 GPU (15GB), this **will crash** with an OOM error.  
We do this intentionally to feel the pain and understand *why* we need QLoRA.

In [ ]:
# ── Load model in full precision (float32 / float16) ─────────────────────────
#
# TinyLlama 1.1B in fp16 needs ~2.2GB just to store weights.
# During training, PyTorch ALSO needs:
#   - Gradients for every parameter  (~2.2GB)
#   - Optimizer states (Adam stores 2 states per param)  (~4.4GB)
#   - Activations for backprop  (~varies, can be huge)
#
# Total ≈ 2.2 + 2.2 + 4.4 = ~9GB MINIMUM, often more.
# Free Colab T4 has 15GB — barely fits TinyLlama, forget 7B models.

from transformers import AutoModelForCausalLM
import torch

print("Loading model in full precision (fp16)...")
print("This is the NAIVE approach — no quantization, all parameters trainable.")
print()

model_full = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

total_params = sum(p.numel() for p in model_full.parameters())
trainable_params = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (100% — this is full fine-tuning)")

In [ ]:
# ── Try to train — EXPECTED TO FAIL WITH OOM ──────────────────────────────────
#
# Expected error:
#   CUDA out of memory. Tried to allocate X MiB.
#   GPU 0 has a total capacity of 14.56 GiB of which Y MiB is free.
#
# WHY IT CRASHES:
#   Forward pass: model sees tokens → produces activations (stored for backprop)
#   Backward pass: computes gradients → needs to store ALL intermediate activations
#   This is called the "activation memory" problem.
#   For 1.1B params at batch_size=2, it easily exceeds 14GB.
#
# LESSON: Full fine-tuning needs A100/H100 (40-80GB) for even 7B models.
#         For free Colab, we need a smarter approach → See PART 5.

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# mlm=False → Causal LM (predict next token), not Masked LM (like BERT)

training_args_full = TrainingArguments(
    output_dir="./full-finetune-attempt",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    fp16=True,
    report_to="none",
    logging_steps=10,
)

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

# ⚠️  This will raise: RuntimeError: CUDA out of memory
# Run it to see the error message, then proceed to the fix below.
try:
    trainer_full.train()
except RuntimeError as e:
    print("\n❌ Expected OOM Error caught:")
    print(str(e)[:300])
    print("\n→ Proceeding to QLoRA solution in PART 5...")

In [ ]:
# ── Clean up GPU memory before next attempt ───────────────────────────────────
#
# Always do this between experiments on Colab.
# gc.collect()        → Python garbage collector frees CPU memory
# empty_cache()       → Releases PyTorch's cached (but unused) GPU memory
#
# NOTE: This does NOT free memory from variables still in scope!
# We must also delete the model reference explicitly.

import gc, torch

del model_full          # remove the reference so Python can free it
del trainer_full
gc.collect()
torch.cuda.empty_cache()

print("✅ GPU memory cleared.")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"   Allocated : {allocated:.2f} GB")
    print(f"   Reserved  : {reserved:.2f} GB")

---
# PART 5 — The Solution: QLoRA (Quantized LoRA)

## What is LoRA?

Instead of updating all 1.1 billion parameters,  
LoRA **freezes the original weights** and adds small **trainable rank-decomposition matrices** (adapters).

```
Original weight:  W  (frozen, huge)
LoRA adds:        W + ΔW  where ΔW = A × B
                  A = (d × r)   B = (r × d)
                  r << d   ← tiny rank!
```

With `r=8`, instead of training `d×d = 2048×2048 = 4M` params per layer,  
you train `2×d×r = 2×2048×8 = 32,768` params — **~120x fewer!**

## What is QLoRA?

QLoRA = LoRA on a **4-bit quantized base model**.

| Technique | Memory Saving | Quality Loss |
|-----------|--------------|-------------|
| fp32 → fp16 | 2x | Tiny |
| fp16 → int8 | 4x | Small |
| fp16 → nf4 (4-bit) | 8x | Acceptable |

**NF4 (NormalFloat4)**: Optimally distributes 4-bit values based on normal distribution of neural network weights. Better than uniform int4.

In [ ]:
# ── Step 1: Define Quantization Config FIRST ─────────────────────────────────
#
# ✅ CORRECT ORDER: Create bnb_config → load model WITH it
#
# ❌ The original notebook did it WRONG:
#    1. Loaded full model (waste of memory)
#    2. Called prepare_model_for_kbit_training on the WRONG model
#    3. Then loaded quantized model (overwriting step 1 and 2)
#    → prepare_model_for_kbit_training was called on a non-quantized model!

from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # Load model in 4-bit
    bnb_4bit_quant_type="nf4",                  # NormalFloat4 — best quality 4-bit format
    bnb_4bit_compute_dtype=torch.float16,       # Compute in fp16 during forward/backward
    bnb_4bit_use_double_quant=True              # Quantize the quantization constants too (saves ~0.4 bits/param extra)
)

print("BitsAndBytesConfig created.")
print("Now we load the model WITH this config applied.")

In [ ]:
# ── Step 2: Load Quantized Model ─────────────────────────────────────────────
#
# device_map={"":0} forces all layers onto GPU 0.
# We use this instead of device_map="auto" during TRAINING
# because "auto" can split layers across CPU+GPU and cause
# tensor device mismatch errors with LoRA adapters.

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"":0}          # All layers → GPU 0 (training-safe)
)

# Required settings for gradient checkpointing to work with quantized models
model.config.use_cache = False          # Gradient checkpointing is incompatible with kv-cache
model.enable_input_require_grads()      # Required for LoRA on quantized models

print("Model loaded in 4-bit!")
mem = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used: {mem:.2f} GB  (vs ~2.2 GB in fp16 — 4x savings!)")

In [ ]:
# ── Step 3: Prepare Model for kbit Training ───────────────────────────────────
#
# This must be called AFTER loading the quantized model.
# It casts layer norms and the output head to fp32 for numerical stability,
# since 4-bit weights can't directly receive gradients.

from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for kbit training.")
print("Layer norms and output head cast to fp32 for stability.")

In [ ]:
# ── Step 4: Define LoRA Configuration ────────────────────────────────────────
#
# r (rank):        How many dimensions the LoRA adapter uses.
#                  Higher r = more expressivity but more parameters.
#                  Typical values: 4, 8, 16, 32, 64
#
# lora_alpha:      Scaling factor for LoRA output.
#                  Effective learning scale = lora_alpha / r
#                  Convention: set lora_alpha = 2 × r
#
# target_modules:  Which attention projections to add LoRA to.
#                  q_proj, v_proj → Minimum (paper default)
#                  + k_proj, o_proj → Better coverage (recommended)
#
# lora_dropout:    Regularization to prevent overfitting on small datasets.
#
# bias="none":     Don't add LoRA to bias terms (standard practice).

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                                                   # Rank
    lora_alpha=16,                                         # Scale = 16/8 = 2.0
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # All attention projections
    lora_dropout=0.05,
    bias="none"
)

qlora_model = get_peft_model(model, lora_config)

# Sanity check — should show ~0.1-1% trainable parameters
qlora_model.print_trainable_parameters()

---
# PART 6 — Training

Now we set up the Trainer with proper arguments.  
Notice several things we were missing before:

| What | Why |
|------|-----|
| `gradient_accumulation_steps` | Simulates larger batch without needing more GPU memory |
| `warmup_steps` | Prevents optimizer from making huge unstable updates at the start |
| `evaluation_strategy` | Tracks validation loss so we can catch overfitting |
| `optim=paged_adamw_8bit` | Memory-efficient optimizer for QLoRA (8-bit Adam) |
| `gradient_checkpointing` | Trade compute for memory — recomputes activations instead of storing them |

In [ ]:
# ── BASELINE: Run inference BEFORE training ───────────────────────────────────
#
# This is something the original notebook completely skipped!
# Without a baseline, you can't measure what fine-tuning actually changed.
#
# Always record:
#   1. What the base model says
#   2. What the fine-tuned model says
#   3. Compare them

test_prompt = "Clinical trials demonstrated that combining Metformin with"

qlora_model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = qlora_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )

baseline_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=" * 60)
print("BASELINE (before fine-tuning):")
print("=" * 60)
print(baseline_output)
print()
print("📝 Save this output — we'll compare it after training.")

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────────

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qlora-finetuned",

    # ── Training schedule ──────────────────────────────────────────────────
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # Effective batch = 2 × 8 = 16
                                     # Larger effective batch → more stable gradients
                                     # without using more memory per step

    # ── Optimizer ──────────────────────────────────────────────────────────
    learning_rate=2e-4,              # QLoRA typically uses higher LR than full fine-tuning
    warmup_steps=50,                 # Gradually increase LR from 0 → 2e-4 over first 50 steps
                                     # Prevents early instability in the optimizer
    optim="paged_adamw_8bit",        # 8-bit Adam with paging → saves ~1.5GB vs regular Adam

    # ── Memory savings ─────────────────────────────────────────────────────
    fp16=True,
    gradient_checkpointing=True,     # Recompute activations during backprop instead of
                                     # storing them. Saves ~60% activation memory.
                                     # Cost: ~33% slower backward pass.

    # ── Evaluation & checkpointing ─────────────────────────────────────────
    evaluation_strategy="steps",     # ✅ We now evaluate on the eval set!
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,     # Automatically use the checkpoint with best eval loss

    # ── Logging ────────────────────────────────────────────────────────────
    logging_steps=50,
    report_to="none",
)

print("Training arguments configured.")

In [ ]:
# ── Setup Trainer ─────────────────────────────────────────────────────────────

from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # Causal LM, not Masked LM
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,    # ✅ Now using our validation split
    data_collator=data_collator,
)

print("Trainer ready!")

In [ ]:
# ── Start Training ────────────────────────────────────────────────────────────
#
# Watch the logs for:
#   train/loss    → should decrease steadily
#   eval/loss     → should also decrease (or stay flat)
#
# ⚠️  WARNING SIGNS:
#   - train/loss keeps dropping but eval/loss starts rising → OVERFITTING
#   - Both losses stay flat → LR too low, or data too similar to pre-train data
#   - Loss oscillates wildly → LR too high, reduce it

qlora_model.train()
trainer.train()

print("\n✅ Training complete!")

In [ ]:
# ── Save the LoRA Adapter ─────────────────────────────────────────────────────
#
# IMPORTANT: We are saving ONLY the LoRA adapter weights, NOT the full model.
# The adapter is tiny (~10-100MB) compared to the base model (~2GB).
# To use it, you load the base model + merge the adapter.

adapter_save_path = "./qlora-finetuned/final-adapter"
qlora_model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

print(f"✅ LoRA adapter saved to: {adapter_save_path}")

# Optional: zip and download from Colab
import zipfile, os

zip_path = "/content/qlora_adapter.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(adapter_save_path):
        for file in files:
            full = os.path.join(root, file)
            arcname = os.path.relpath(full, adapter_save_path)
            zf.write(full, arcname)

print(f"✅ Adapter zipped to: {zip_path}")
print("   (Right-click → Download from the Colab file browser)")

---
# PART 7 — Inference: Compare Before vs After

In [ ]:
# ── Load Base Model + LoRA Adapter for Inference ─────────────────────────────
#
# ❌ ORIGINAL NOTEBOOK MISTAKE:
#    Training used device_map={"":0} but inference used device_map="auto".
#    "auto" can put some layers on CPU and some on GPU → tensor device mismatch!
#
# ✅ FIX: Keep device_map consistent.
#    For inference, "auto" is fine as long as you're not continuing to train.
#    But if you ran training with device_map={"":0}, load inference the same way
#    to be safe.

from peft import PeftModel

# Load base model (we can use quantization here too, or load in fp16)
base_model_infer = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"":0}           # ✅ Same as training — consistent device
)

# Load LoRA adapter on top
fine_tuned_model = PeftModel.from_pretrained(base_model_infer, adapter_save_path)
fine_tuned_model.eval()

print("Fine-tuned model loaded for inference.")

In [ ]:
# ── Compare Base Model vs Fine-Tuned Model ────────────────────────────────────

def generate(model, prompt: str, max_new_tokens: int = 100) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


test_prompts = [
    "Clinical trials demonstrated that combining Metformin with",
    "The mechanism of action of Metformin involves",
    "Common side effects observed in patients taking Metformin include",
]

for prompt in test_prompts:
    print("=" * 65)
    print(f"PROMPT: {prompt}")
    print("-" * 65)
    print("BEFORE (baseline saved earlier):")
    print(baseline_output[:200])      # we saved this before training
    print()
    print("AFTER (fine-tuned):")
    print(generate(fine_tuned_model, prompt))
    print()

---
# PART 8 — Summary: What We Learned

| Step | ❌ Wrong Approach | ✅ Fixed Approach | Why It Matters |
|------|------------------|-----------------|----------------|
| **Chunking** | Double-newline split | Fixed-size sliding window with overlap | Consistent input length, no sentence cuts |
| **Labels** | `labels = input_ids` (includes padding) | Mask padding with `-100` | Padding should NOT contribute to loss |
| **Dataset split** | No validation set | 90/10 train/eval split | Can't detect overfitting without eval |
| **Model loading order** | Load full model → quantize later | Create `bnb_config` → load quantized directly | Calling `prepare_model_for_kbit_training` on a non-quantized model is wrong |
| **Full fine-tune** | Tried on free Colab (OOM crash) | QLoRA — train only LoRA adapters | 1.1B model needs ~9GB just for gradients |
| **LoRA target modules** | Only `q_proj, v_proj` | All 4: `q_proj, k_proj, v_proj, o_proj` | Better gradient coverage, faster convergence |
| **Batch size** | `batch_size=2` only | `batch_size=2` + `gradient_accumulation=8` | Effective batch=16, more stable training |
| **LR warmup** | No warmup | `warmup_steps=50` | Prevents unstable early updates |
| **device_map** | Train `{"":0}` / Infer `auto` (mismatch) | Both `{"":0}` | Avoids tensor device mismatch errors |
| **Baseline** | No baseline comparison | Run inference before training | Can't measure improvement without it |

---

## Next Steps to Explore

1. **Merge adapter into base model** — for faster inference, merge LoRA weights permanently  
   ```python
   merged = fine_tuned_model.merge_and_unload()
   merged.save_pretrained("./merged-model")
   ```

2. **Increase `r` for better quality** — try `r=16` or `r=32` if you have more VRAM

3. **Try Unsloth** — 2x faster than this setup, same accuracy, great for Colab

4. **Instruction fine-tuning** — use `trl.SFTTrainer` with `{instruction, response}` dataset format

5. **Push to HuggingFace Hub**  
   ```python
   fine_tuned_model.push_to_hub("your-username/your-model-name")
   ```